# Finding rate RNN fixed points

In [1]:
import os, scipy.io 
import numpy as np 
# %matplotlib ipympl
import matplotlib.pyplot as plt 
from mpl_toolkits.mplot3d import axes3d
import sys

sys.path.append('/home/nuttidalab/Documents/spikeRNN/rate/')
# import model
from model import generate_input_stim_xor, eval_tf

sys.path.append('/home/nuttidalab/Documents/spikeRNN/analysis_code/utils/')
import vis_utils as vu
import importlib

sys.path.append('/home/nuttidalab/Documents/spikeRNN/analysis_code/trajectory/fixed_pt/')
import load_RNN_model as lrm

2025-02-26 09:23:53.998419: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-26 09:23:54.007920: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-02-26 09:23:54.017395: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-02-26 09:23:54.020221: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-26 09:23:54.028195: I tensorflow/core/platform/cpu_feature_guar

Instructions for updating:
non-resource variables are not supported in the long term


/home/nuttidalab/anaconda3/envs/rnn_test/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [2]:
models_type = 'good_models'

models_dir = '/scratch/spikeRNN/models/DMS_OSF/' # dir where all models are located
results_dir = f'{models_dir}{models_type}/' # dir where results are saved

model_list_path = f'{results_dir}{models_type}_list.mat' # list (saved from matlab) of models of interest
model_list_cell = scipy.io.loadmat(model_list_path)['stable_mods'][0] 
model_list = [model_list_cell[i][0] for i in range(len(model_list_cell))]
RNN_model_file = model_list[0] # just grab the first model

print(f'Loading model: {RNN_model_file}')

Loading model: Task_xor_N_200_Taus_4.0_25.0_Act_sigmoid_2019_09_06_152659.mat


In [3]:
rnn_path = os.path.join(models_dir, RNN_model_file) # double check this!!!
model = lrm.load_RNN_model(rnn_path)

In [4]:
print(rnn_path)
print(model)

/scratch/spikeRNN/models/DMS_OSF/Task_xor_N_200_Taus_4.0_25.0_Act_sigmoid_2019_09_06_152659.mat
tRNN(2, 200)


In [5]:
settings = {
        'T': 500, # trial duration (in steps)
        'stim_on': 200, # input stim onset (in steps)
        'stim_dur': 50, # input stim duration (in steps)
        'delay': 10, # delay b/w the two stimuli (in steps)
        'DeltaT': 1, # sampling rate
        'taus': 20, # decay time-constants (in steps)
        'task': 'xor', # task name
        }

In [6]:
def generate_xor_type(T, stim_on, stim_dur, delay, stim1=1, stim2=-1):
     
     u = np.zeros((2, T))
     
     u[0, stim_on:stim_on+stim_dur] = stim1
     u[1, stim_on+stim_dur+delay:stim_on+2*stim_dur+delay] = stim2

     if stim1 == stim2:
          label = 'same'
     else: 
          label = 'diff'
     
     return u, label

In [7]:
model_results_dir = f'{models_dir}{RNN_model_file[:-4]}/'
print(model_results_dir)

/scratch/spikeRNN/models/DMS_OSF/Task_xor_N_200_Taus_4.0_25.0_Act_sigmoid_2019_09_06_152659/


In [ ]:
n_trials = 100
stim1s = [-1, 1]
stim2s = [-1, 1]
n_trials_per_condition = int(n_trials/(len(stim1s)*len(stim2s)))

synX = np.zeros((n_trials, settings['T'], 200)) # nTrials x nTimes x nNeurons
trial_stim_labels = np.zeros((n_trials,2))
for i, stim1 in enumerate(stim1s):
    for j, stim2 in enumerate(stim2s):
        for k in range(n_trials_per_condition):
            trial_idx = (i*len(stim2s)+j)*n_trials_per_condition + k

            u, label = generate_xor_type(settings['T'], settings['stim_on'], settings['stim_dur'], settings['delay'], stim1, stim2)
            x, r, o, _ = eval_tf(model_dir=rnn_path, settings=settings, u=u)

            synX[trial_idx,:,:] = x.T
            trial_stim_labels[trial_idx,0] = stim1
            trial_stim_labels[trial_idx,1] = stim2

# np.save(f'{model_results_dir}synX.npy', synX)
# np.save(f'{model_results_dir}trial_stim1_labels.npy', trial_stim1_labels)

In [ ]:
# np.save(f'{model_results_dir}synX.npy', synX)
# np.save(f'{model_results_dir}trial_stim1_labels.npy', trial_stim1_labels)

In [75]:
xx_trials = np.load(f'{model_results_dir}synX.npy')
trial_stim_labels = np.load(f'{model_results_dir}trial_stim_labels.npy')


In [28]:
stim_on = 200
stim_dur = 50
delay=10
T = 500  

In [84]:
task_type = 1
delay_times = np.arange(stim_on+stim_dur,stim_on+stim_dur+delay)
xx_trials_delay = xx_trials[trial_stim_labels[:,0] == task_type,:,:][:,delay_times,:]

xx_trials_delay.shape

(50, 10, 200)

In [78]:
n_initial = 200
initial_states = np.zeros((n_initial,xx_trials_delay.shape[2])) # n_initial x nNeurons
# initial_states2 = np.zeros((n_initial,xx_trials_delay.shape[2]))
for iI in np.arange(n_initial):
    rand_trial = np.random.randint(xx_trials_delay.shape[0]) 		
    rand_time = np.random.randint(xx_trials_delay.shape[1])
    initial_states[iI,:] = xx_trials_delay[rand_trial,-1,:]				

In [89]:
sys.path.append('/home/nuttidalab/Documents/spikeRNN/analysis_code/utils/fixed-point-finder')
from FixedPointFinderTorch import FixedPointFinderTorch as FixedPointFinder
from plot_utils import plot_fps # in fixedpointfinder folder

In [92]:
model

tRNN(2, 200)

In [93]:
initial_states.shape

(200, 200)

In [94]:
inputs.shape

(2, 500)

In [95]:
inputs = np.zeros((2, T))

# Choosing input vector u for each trial type/modality type		
inputs[0, stim_on:stim_on+stim_dur] = task_type
inputs[1, stim_on+stim_dur+delay:stim_on+2*stim_dur+delay] = task_type

inputs = np.zeros((1, T))
inputs[0, stim_on:stim_on+stim_dur] = task_type
inputs[0, stim_on+stim_dur+delay:stim_on+2*stim_dur+delay] = task_type

        
fpf_hps = {
			'max_iters': 30000,
			'lr_init': 0.1,
			'outlier_distance_scale': 10.0,			
			'tol_unique':10, #75
			'verbose': True, 
			'super_verbose': True}

# Defining and running fixed point optimization
fpf = FixedPointFinder(model, **fpf_hps)
unique_fps,all_fps = fpf.find_fixed_points(initial_states, inputs)    
fp_dict = {'xstar': unique_fps.xstar, 'is_stable':unique_fps.is_stable, \
        'J_xstar':unique_fps.J_xstar, 'eigval_J_xstar': unique_fps.eigval_J_xstar, \
            'eigvec_J_xstar':unique_fps.eigvec_J_xstar}


Searching for fixed points from 200 initial states.

	Freezing model parameters so model is not affected by fixed point optimization.
	Finding fixed points via joint optimization.


RuntimeError: mat1 and mat2 shapes cannot be multiplied (200x2 and 500x200)

In [98]:
model.taus_sig.shape

torch.Size([200, 1])

In [2]:
models_dir = '/scratch/spikeRNN/models/DMS_OSF/' # dir where all models are located
model_fname = 'Task_xor_N_200_Taus_4.0_25.0_Act_sigmoid_2019_09_06_162018'
model_path = os.path.join(models_dir, model_fname)
mat_data = scipy.io.loadmat(model_path)

In [3]:
mat_data.keys()

dict_keys(['__header__', '__version__', '__globals__', 'N', 'abc', 'activation', 'all_perfs', 'b_out', 'curr_mod', 'eval_labels', 'eval_loss_mean', 'eval_os', 'eval_perf_mean', 'exc', 'inh', 'losses', 'm', 'model_path', 'o', 'opt_scaling_factor', 'r', 'r0', 'scaling_factors', 'som_N', 'som_m', 'stability', 'stable_mods2', 'stable_outs', 'stable_perfs', 'stable_trials', 'target', 'task_path5', 'tau', 'taus', 'taus_gaus', 'taus_gaus0', 'tr', 'u', 'w', 'w0', 'w_in', 'w_in0', 'w_out', 'x', 'x0', 'auto_N', 'auto_N_fr', 'auto_c', 'mean_decay', 'new_auto_c', 'outs', 'syn_decay', 'taus_decay_ms', 'trial_spks', 'use_this_W', 'pref_stim'])

In [4]:
mat_data = scipy.io.loadmat(model_path)
net = model.FR_RNN_dale(200, 0.2, 0.2, 
                        mat_data['w_in'], mat_data['som_N'], 'gaus', 
                        1.5, True, mat_data['w_out'])
net.load_net(model_path)
net.display()

Network Settings
Number of Units:  200
	 Number of Excitatory Units:  163
	 Number of Inhibitory Units:  37
Weight Matrix, W
	 Zero Weights: 100.00 %
	 Positive Weights: 0.00 %
	 Negative Weights: 0.01 %


In [10]:
importlib.reload(lrm)
model = lrm.load_RNN_model(model_path)
model

tRNN(2, 200)

In [ ]:
model